# Tobit model: inference of binding-affinity landscapes

Self-contained reimplementation of the censored regression in the Methods; covers genotype encoding and left-censoring, the interaction-only polynomial basis, the Tobit log-likelihood with its analytic gradient and elastic-net penalty, penalized maximum likelihood, standard errors from the inverse observed information of the unpenalized likelihood, prediction over all 1024 genotypes, leave-one-genotype-out cross-validation scored by summed held-out NLL, and selection over the order by penalty grid.

The final cells refit 1NC9 against CH505TF at the published setting and compare against the shipped coefficients: 386 retained terms, r = 0.9989, maximum absolute difference 0.137. `max_folds` runs a handful of folds here to test a portion of the overall cluster jobs submitted.

## Setup

In [ ]:
import itertools
import os

import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.special import log_ndtr
from scipy.stats import norm
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures

## Configuration

In [ ]:
data_dir = "../data"

site_cols = [f"site_{i}" for i in range(10)]
kd_column = "nlog10_Kd"
antigen_col = "antigen"
genotype_id_column = "id"       # a LOGO fold holds out every replicate of one id
germline_token, mutant_token = "G", "M"

left_censor = 6.0
right_censor = -1.0             # -1 disables the right side

orders = [1, 2, 3, 4, 5]        # order 6 is near-saturated at ~850 genotypes and did not converge
alphas_l1 = [0.0, 0.001, 0.01, 0.1, 1.0, 10.0]
alphas_l2 = [0.0, 0.001, 0.01, 0.1, 1.0, 10.0]

# the objective sums rather than averages the per-point NLL, so the penalty is
# rescaled by the number of penalized terms to keep alpha comparable across orders
intercept_bounds = (0.0, 14.0)  # -log10 Kd cannot be negative
coef_bounds = (-14.0, 14.0)
log_sigma_lower = -10.0
l1_smoothing = 1e-6
hessian_step = 1e-5

log_sqrt_2pi = 0.5 * np.log(2.0 * np.pi)

## Data preparation

In [ ]:
def encode_genotypes(df):
    """Site tokens to a 0/1 matrix, germline as 0 and mature as 1."""
    token_map = {germline_token: 0, mutant_token: 1}
    try:
        return np.vectorize(token_map.__getitem__)(df[site_cols].to_numpy()).astype(float)
    except KeyError as exc:
        raise ValueError(f"unexpected site token {exc}, expected only G and M")


def apply_censoring(y):
    """Censor indicator (-1 left, 0 observed, 1 right) with the response clamped to the threshold."""
    y = np.asarray(y, float).copy()
    cens = np.zeros(len(y))
    if left_censor > -1:
        cens[y <= left_censor] = -1
        y[cens == -1] = left_censor
    if right_censor > -1:
        cens[y >= right_censor] = 1
        y[cens == 1] = right_censor
    return y, cens


def prepare(df, antigen):
    """One antigen's rows as design-ready arrays plus the labels that define LOGO folds."""
    sub = df.loc[df[antigen_col] == antigen].reset_index(drop=True)
    if sub.empty:
        raise ValueError(f"no rows for antigen {antigen}")
    genotypes = encode_genotypes(sub)
    y, cens = apply_censoring(sub[kd_column].to_numpy(float))
    return genotypes, y, cens, sub[genotype_id_column].tolist()

## Design matrix

Terms represent interactions among distinct sites only. Columns present in zero or one genotype carry no information about an interaction and are dropped.

In [ ]:
def build_design(genotypes, order):
    """Interaction-only polynomial basis, dropping terms seen in 0 or 1 genotypes."""
    poly = PolynomialFeatures(order, interaction_only=True, include_bias=True)
    full = poly.fit_transform(genotypes)

    name_map = {f"x{i}": site_cols[i] for i in range(len(site_cols))}
    name_map["1"] = "intercept"
    nested = [[name_map[tok] for tok in n.split(" ")] for n in poly.get_feature_names_out()]
    names = [",".join(parts) for parts in nested]
    term_orders = np.array([0 if parts == ["intercept"] else len(parts) for parts in nested])

    col_sums = full.sum(axis=0)
    keep = np.where((col_sums != 0) & (col_sums != 1))[0]
    return poly, keep, [names[i] for i in keep], term_orders[keep]

## Likelihood and gradient

In [ ]:
def normal_logpdf(z):
    """Standard normal log density, faster than scipy inside the optimizer loop."""
    return -log_sqrt_2pi - 0.5 * z * z


def split_by_censoring(x, y, cens):
    """Design and response split into left-censored, observed and right-censored blocks."""
    blocks = []
    for code_value in (-1, 0, 1):
        m = cens == code_value
        blocks.append((x[m], y[m]) if m.any() else (None, None))
    return blocks


def tobit_nll(blocks, params):
    """Unpenalized Tobit negative log-likelihood, params = (beta, log sigma)."""
    b, log_s = params[:-1], params[-1]
    s = np.exp(log_s)
    (x_left, y_left), (x_mid, y_mid), (x_right, y_right) = blocks

    total = 0.0
    if y_left is not None:
        total += norm.logcdf((y_left - x_left @ b) / s).sum()
    if y_right is not None:
        total += norm.logcdf((x_right @ b - y_right) / s).sum()
    if y_mid is not None:
        total += (normal_logpdf((y_mid - x_mid @ b) / s) - log_s).sum()
    return -total


def elastic_penalty(b, alpha_l1, alpha_l2):
    """Elastic-net penalty on the non-intercept coefficients, L1 smoothed."""
    reg = b[1:]
    out = 0.0
    if alpha_l2 > 0:
        out += alpha_l2 * np.sum(reg ** 2)
    if alpha_l1 > 0:
        out += alpha_l1 * np.sum(np.sqrt(reg ** 2 + l1_smoothing))
    return out


def tobit_nll_grad(blocks, params, alpha_l1=0.0, alpha_l2=0.0):
    """Analytic gradient of the penalized objective in (beta, log sigma)."""
    b, log_s = params[:-1], params[-1]
    s = np.exp(log_s)
    (x_left, y_left), (x_mid, y_mid), (x_right, y_right) = blocks

    beta_jac = np.zeros(len(b))
    sigma_jac = 0.0
    if y_left is not None:
        stats = (y_left - x_left @ b) / s
        frac = np.exp(normal_logpdf(stats) - log_ndtr(stats))   # inverse Mills ratio in log space
        beta_jac -= frac @ (x_left / s)
        sigma_jac -= frac @ stats
    if y_right is not None:
        stats = (x_right @ b - y_right) / s
        frac = np.exp(normal_logpdf(stats) - log_ndtr(stats))
        beta_jac += frac @ (x_right / s)
        sigma_jac -= frac @ stats
    if y_mid is not None:
        stats = (y_mid - x_mid @ b) / s
        beta_jac += stats @ (x_mid / s)
        sigma_jac += (stats ** 2 - 1).sum()

    if alpha_l2 > 0:
        beta_jac[1:] -= 2 * alpha_l2 * b[1:]
    if alpha_l1 > 0:
        beta_jac[1:] -= alpha_l1 * (b[1:] / np.sqrt(b[1:] ** 2 + l1_smoothing))

    return -np.append(beta_jac, sigma_jac)

## Fitting

In [ ]:
def observed_information_stderr(blocks, params):
    """Standard errors from the inverse observed information of the UNPENALIZED likelihood.

    The penalty is a fitting device, so including it would understate the uncertainty.
    The Hessian is taken by central differences on the analytic gradient.
    """
    n_p = len(params)
    hess = np.zeros((n_p, n_p))
    for i in range(n_p):
        up, down = params.copy(), params.copy()
        up[i] += hessian_step
        down[i] -= hessian_step
        hess[:, i] = (tobit_nll_grad(blocks, up) - tobit_nll_grad(blocks, down)) / (2 * hessian_step)
    hess = 0.5 * (hess + hess.T)

    try:
        diag = np.diag(np.linalg.inv(hess))
    except np.linalg.LinAlgError:
        return np.full(n_p - 1, np.nan)
    return np.sqrt(np.where(diag > 0, diag, np.nan))[:-1]   # drop the log sigma entry


def fit_tobit(x, y, cens, alpha_l1=0.0, alpha_l2=0.0):
    """Penalized Tobit MLE by L-BFGS-B, started from the OLS solution."""
    blocks = split_by_censoring(x, y, cens)
    ols = LinearRegression(fit_intercept=False).fit(x, y)
    start = np.append(ols.coef_, np.log(np.sqrt(np.var(y - ols.predict(x)))))
    bounds = [intercept_bounds] + [coef_bounds] * (x.shape[1] - 1) + [(log_sigma_lower, None)]

    result = minimize(
        lambda p: tobit_nll(blocks, p) + elastic_penalty(p[:-1], alpha_l1, alpha_l2),
        start, method="L-BFGS-B", bounds=bounds,
        jac=lambda p: tobit_nll_grad(blocks, p, alpha_l1, alpha_l2),
    )
    return result.x[:-1], float(np.exp(result.x[-1])), observed_information_stderr(blocks, result.x), result


def pointwise_nll(x, y, cens, coeffs, sigma):
    """Per-observation unpenalized Tobit NLL at fitted parameters, for scoring held-out points."""
    mu = x @ coeffs
    out = np.zeros(len(y))
    mid, left, right = cens == 0, cens == -1, cens == 1
    if mid.any():
        out[mid] = -(normal_logpdf((y[mid] - mu[mid]) / sigma) - np.log(sigma))
    if left.any():
        out[left] = -norm.logcdf((y[left] - mu[left]) / sigma)
    if right.any():
        out[right] = -norm.logcdf((mu[right] - y[right]) / sigma)
    return out

## Full-data fit and landscape prediction

`predicted_kd_uncensored` is the latent phenotype (read by the Walsh
Hadamard decomposition and the maturation simulations).
`predicted_kd_full` is that value clamped at the floor, matching what the assay
could have reported.

In [ ]:
def fit_complex(df, antigen, order, alpha_l1=0.0, alpha_l2=0.0):
    """Full-data fit for one antibody-antigen complex. Returns (df_params, df_all, sigma)."""
    genotypes, y, cens, _ = prepare(df, antigen)
    poly, keep, names, term_orders = build_design(genotypes, order)
    x = poly.transform(genotypes)[:, keep]
    scale = max(1, len(keep) - 1)

    coeffs, sigma, stderr, _ = fit_tobit(x, y, cens, alpha_l1 * scale, alpha_l2 * scale)
    df_params = pd.DataFrame({"term": names, "order": term_orders.astype(int),
                              "coefficient": coeffs, "std_error": stderr})

    grid = np.array(list(itertools.product([0, 1], repeat=len(site_cols))), float)
    latent = poly.transform(grid)[:, keep] @ coeffs
    df_all = pd.DataFrame({
        "genotype_binary": ["".join(str(int(g)) for g in r) for r in grid],
        "genotype_tokens": ["".join(mutant_token if g else germline_token for g in r) for r in grid],
        "predicted_kd_full": np.maximum(latent, left_censor) if left_censor > -1 else latent,
        "predicted_kd_uncensored": latent,
    })
    for i, col in enumerate(site_cols):
        df_all[col] = np.where(grid[:, i] == 1, mutant_token, germline_token)
    return df_params, df_all, sigma

## Cross-validation and model selection

In [ ]:
def resolve_reg_type(alpha_l1, alpha_l2):
    """Which regime a grid cell lands in, so the alpha = 0 edges are not mislabeled elastic."""
    if alpha_l1 > 0 and alpha_l2 > 0:
        return "elastic_net"
    if alpha_l1 > 0:
        return "l1"
    if alpha_l2 > 0:
        return "l2"
    return "none"


def logo_cv(df, antigen, order, alpha_l1=0.0, alpha_l2=0.0, max_folds=None,
            skip_failed_folds=False, verbose=False):
    """Summed held-out Tobit NLL over leave-one-genotype-out folds for one condition."""
    genotypes, y, cens, labels = prepare(df, antigen)
    poly, keep, _, _ = build_design(genotypes, order)
    x = poly.transform(genotypes)[:, keep]
    scale = max(1, len(keep) - 1)

    unique = list(dict.fromkeys(labels))
    n_folds = len(unique) if max_folds is None else min(max_folds, len(unique))
    label_arr = np.array(labels, dtype=object)

    fold_nlls, n_failed, first_error = [], 0, None
    for i in range(n_folds):
        test = label_arr == unique[i]
        try:
            coeffs, sigma, _, _ = fit_tobit(x[~test], y[~test], cens[~test],
                                            alpha_l1 * scale, alpha_l2 * scale)
            value = float(pointwise_nll(x[test], y[test], cens[test], coeffs, sigma).sum())
            if not np.isfinite(value):
                raise ValueError(f"non-finite nll on fold {i}")
            fold_nlls.append(value)
        except Exception as exc:
            n_failed += 1
            if first_error is None:
                first_error = f"fold {i}: {type(exc).__name__}: {exc}"
            if verbose:
                print("fold", i, "failed:", exc)

    if n_failed and not skip_failed_folds:
        cv_nll = float("nan")
    elif not fold_nlls:
        cv_nll = float("nan")
    else:
        cv_nll = float(np.sum(fold_nlls))

    return {"antigen": antigen, "order": order, "alpha_l1": alpha_l1, "alpha_l2": alpha_l2,
            "reg_type": resolve_reg_type(alpha_l1, alpha_l2), "n_folds": n_folds,
            "n_folds_ok": len(fold_nlls), "n_folds_failed": n_failed,
            "cv_nll": cv_nll, "first_error": first_error}


def sweep(df, antigen, out_path=None, max_folds=None, order_list=None):
    """Run the order by penalty grid for one complex, appending each condition as it finishes."""
    rows = []
    for order in (order_list or orders):
        for a1 in alphas_l1:
            for a2 in alphas_l2:
                row = logo_cv(df, antigen, order, a1, a2, max_folds=max_folds)
                rows.append(row)
                if out_path:
                    header = not os.path.exists(out_path) or os.path.getsize(out_path) == 0
                    with open(out_path, "a", newline="") as fh:
                        pd.DataFrame([row]).to_csv(fh, header=header, index=False)
                        fh.flush()
                        os.fsync(fh.fileno())
                print(f"order {order} a1={a1:g} a2={a2:g} cv_nll={row['cv_nll']:.4f}", flush=True)
    return pd.DataFrame(rows)


def select_best(results):
    """Winning condition per antigen: lowest cv_nll among conditions whose folds all completed."""
    tab = pd.DataFrame(results)
    clean = tab[tab["cv_nll"].notna() & (tab["n_folds_failed"] == 0)]
    if clean.empty:
        return tab, clean
    return tab, clean.sort_values("cv_nll").groupby("antigen", as_index=False).first()

## Check against the established fit

1NC9 against CH505TF, refit at the published best-fit setting of order 4 with
both penalties at 0.1, compared against `1NC9_CH505TF_BEST_df_params.csv`.

In [ ]:
kds = pd.read_csv(f"{data_dir}/1NC9_kds.csv")
params_new, all_new, sigma = fit_complex(kds, "CH505TF", order=4, alpha_l1=0.1, alpha_l2=0.1)
print(f"{len(params_new)} retained terms, sigma = {sigma:.4f}")

ref = pd.read_csv(f"{data_dir}/1NC9_CH505TF_BEST_df_params.csv")
both = params_new.merge(ref, on=["term", "order"], suffixes=("_new", "_ref"))
r = np.corrcoef(both.coefficient_new, both.coefficient_ref)[0, 1]
print(f"matched {len(both)} terms, coefficient r = {r:.4f}, "
      f"max abs diff = {(both.coefficient_new - both.coefficient_ref).abs().max():.4f}")

display(both[both.order <= 1][["term", "coefficient_new", "coefficient_ref"]].round(4))

## Cross-validation

In [ ]:
row = logo_cv(kds, "CH505TF", order=2, alpha_l1=0.01, alpha_l2=0.01, max_folds=12)
display(pd.DataFrame([row]))